# 07 — Migration Reconciliation Report

Reads the defect log produced by **06 — Migration Reconciliation** and generates a
formal, stakeholder-facing reconciliation report.

Mirrors the deliverable format a QA analyst hands to engineering: executive summary,
reconciliation scorecard (vs the injected manifest), defect breakdown by severity
and type, and a remediation section. The report is printed inline **and** written
to a Delta table so every run is versioned and auditable.

**Inputs**
- `dq.migration_defects` — defect log from 06
(schema: `defect_id, run_id, table_name, column_name, defect_type,`
`source_value, target_value, severity, root_cause_hypothesis, detected_at`)
- `raw.cr_migration_manifest` — injected-defect ground truth from 05
(schema: `defect_class, description, expected_count, generated_at`)

**Outputs**
- Printed reconciliation report (stakeholder format)
- `dq.reconciliation_reports` — the report persisted, keyed by run_id

## Config
Reuses the same path helpers 05 / 06 use (`r(...)` for raw, `dq(...)` for dq) if
they are already defined in the session. Falls back to a standalone config load so
07 can also be run on its own.

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

# 05/06 define r() and dq() helpers. Reuse them if present; otherwise rebuild
# from the shared pipeline_config so 07 is runnable standalone too.
try:
    DEFECTS_TBL  = dq("migration_defects")
    MANIFEST_TBL = r("cr_migration_manifest")
    REPORT_TBL   = dq("reconciliation_reports")
    SRC_TBL      = r("cr_businesses_source")
    TGT_TBL      = r("cr_businesses_target")
    print("Reusing r()/dq() helpers from 05/06 session.")
except NameError:
    cfg = spark.table("workspace.fieldops_audit.pipeline_config").collect()[0].asDict()
    CATALOG = cfg.get("catalog", "workspace")
    RAW     = cfg.get("raw_schema", cfg.get("raw", "fieldops_raw"))
    DQ      = cfg.get("dq_schema",  cfg.get("dq",  "fieldops_dq"))
    DEFECTS_TBL  = f"{CATALOG}.{DQ}.migration_defects"
    MANIFEST_TBL = f"{CATALOG}.{RAW}.cr_migration_manifest"
    REPORT_TBL   = f"{CATALOG}.{DQ}.reconciliation_reports"
    SRC_TBL      = f"{CATALOG}.{RAW}.cr_businesses_source"
    TGT_TBL      = f"{CATALOG}.{RAW}.cr_businesses_target"
    print("Standalone config load from workspace.fieldops_audit.pipeline_config")

print(f"  defects  : {DEFECTS_TBL}")
print(f"  manifest : {MANIFEST_TBL}")
print(f"  report   : {REPORT_TBL}")

Standalone config load from workspace.fieldops_audit.pipeline_config
  defects  : workspace.fieldops_dq.migration_defects
  manifest : workspace.fieldops_raw.cr_migration_manifest
  report   : workspace.fieldops_dq.reconciliation_reports


## Select the run to report on
Defaults to the most recent run in the defect log. Set `RUN_ID` to regenerate a
report for an earlier run (the defect log from 06 is overwritten per run, so
normally there is one run present — this still works either way).

In [0]:
RUN_ID = None  # set to a specific run_id to report on an earlier run

defects = spark.table(DEFECTS_TBL)

if RUN_ID is None:
    RUN_ID = (
        defects.select("run_id")
        .orderBy(F.col("run_id").desc())
        .limit(1)
        .collect()[0]["run_id"]
    )

run_defects = defects.filter(F.col("run_id") == RUN_ID)
n_defects = run_defects.count()

if n_defects == 0:
    raise ValueError(
        f"No defects found for run_id={RUN_ID}. Run notebook 06 first."
    )

print(f"Reporting on run: {RUN_ID}")
print(f"Defects in this run: {n_defects}")

Reporting on run: MIG-20260518-031539-70BA48
Defects in this run: 13


## Severity & defect-type breakdown
Two rollups stakeholders care about: how bad (severity) and what kind
(`defect_type`, the categorical classification 06 assigns).

In [0]:
SEVERITY_ORDER = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}

sev_rows = run_defects.groupBy("severity").agg(F.count("*").alias("n")).collect()
sev_counts = {row["severity"]: row["n"] for row in sev_rows}
sev_counts = dict(
    sorted(sev_counts.items(), key=lambda kv: SEVERITY_ORDER.get(kv[0], 99))
)

type_rows = (
    run_defects.groupBy("defect_type")
    .agg(F.count("*").alias("n"))
    .orderBy(F.col("n").desc())
    .collect()
)

print("Severity breakdown:")
for sev, n in sev_counts.items():
    print(f"  {sev:<9}: {n}")

print("\nDefect-type breakdown:")
for row in type_rows:
    print(f"  {row['defect_type']:<28}: {row['n']}")

Severity breakdown:
  CRITICAL : 2
  HIGH     : 5
  MEDIUM   : 4
  LOW      : 2

Defect-type breakdown:
  FIELD_VALUE_MISMATCH        : 4
  SCHEMA_DRIFT_COLUMN_ADDED   : 2
  ROW_COUNT_MISMATCH          : 1
  SCHEMA_DRIFT_COLUMN_DROPPED : 1
  ROWS_LOST_IN_MIGRATION      : 1
  DUPLICATE_KEYS_IN_TARGET    : 1
  ROW_VALUE_MISMATCH          : 1
  AGGREGATE_SUM_MISMATCH      : 1
  CATEGORICAL_DOMAIN_DRIFT    : 1


## Reconciliation scorecard (vs injected manifest)
The manifest (`defect_class`) is the ground truth of what 05 deliberately broke.
Each manifest class maps to one or more `defect_type` values that 06 logs. A class
is CAUGHT if at least one logged defect of a mapped type exists. Detection rate is
the headline number of the report.

In [0]:
# Manifest defect_class  ->  defect_type values 06 logs for that class.
CLASS_TO_TYPES = {
    "row_loss":      {"ROWS_LOST_IN_MIGRATION"},
    "duplicates":    {"DUPLICATE_KEYS_IN_TARGET"},
    "schema_drift":  {"SCHEMA_DRIFT_COLUMN_DROPPED",
                      "SCHEMA_DRIFT_COLUMN_ADDED",
                      "SCHEMA_DRIFT_TYPE_CHANGED"},
    "numeric_drift": {"AGGREGATE_SUM_MISMATCH"},
    "taxonomy":      {"CATEGORICAL_DOMAIN_DRIFT"},
    "corruption":    {"FIELD_VALUE_MISMATCH", "ROW_HASH_MISMATCH"},
}

manifest = spark.table(MANIFEST_TBL)
injected_classes = [
    row["defect_class"] for row in manifest.select("defect_class").distinct().collect()
]

logged_types = {row["defect_type"] for row in type_rows}

scorecard = []
for cls in injected_classes:
    mapped = CLASS_TO_TYPES.get(cls, set())
    hit = bool(mapped & logged_types)
    scorecard.append((cls, hit))

scorecard.sort(key=lambda x: x[0])
n_caught = sum(1 for _, hit in scorecard if hit)
n_total = len(scorecard)
detection_rate = f"{n_caught}/{n_total}"

print("── RECONCILIATION SCORECARD (vs injected manifest) ───")
for cls, hit in scorecard:
    print(f"  {cls:<14} {'✓ CAUGHT' if hit else '✗ MISSED'}")
print(f"\n  Detection rate: {detection_rate} injected defect classes")

── RECONCILIATION SCORECARD (vs injected manifest) ───
  corruption     ✓ CAUGHT
  duplicates     ✓ CAUGHT
  numeric_drift  ✓ CAUGHT
  row_loss       ✓ CAUGHT
  schema_drift   ✓ CAUGHT
  taxonomy       ✓ CAUGHT

  Detection rate: 6/6 injected defect classes


## Build the report document
Assembles the stakeholder report. Structure mirrors the Data Compass remediation
report: Executive Summary → Scope → Findings → Scorecard → Defect Detail →
Remediation → Conclusion.

In [0]:
now_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

detail_rows = (
    run_defects
    .withColumn(
        "_sev_ord",
        F.when(F.col("severity") == "CRITICAL", 0)
         .when(F.col("severity") == "HIGH", 1)
         .when(F.col("severity") == "MEDIUM", 2)
         .otherwise(3),
    )
    .orderBy("_sev_ord", "defect_type")
    .select("defect_id", "severity", "defect_type", "column_name",
            "source_value", "target_value", "root_cause_hypothesis")
    .collect()
)

# source/target table names resolved in the config cell
src_tbl = SRC_TBL
tgt_tbl = TGT_TBL

lines = []
def w(s=""):
    lines.append(s)

w("=" * 70)
w("  MIGRATION RECONCILIATION REPORT")
w("=" * 70)
w(f"  Run ID        : {RUN_ID}")
w(f"  Generated     : {now_utc}")
w(f"  Source        : {src_tbl}")
w(f"  Target        : {tgt_tbl}")
w(f"  Defects logged: {n_defects}")
w(f"  Detection rate: {detection_rate} injected defect classes")
w("=" * 70)
w()

w("1. EXECUTIVE SUMMARY")
w("-" * 70)
w("  A source-to-target reconciliation validated the migration of the CR")
w("  businesses registry into the Databricks lakehouse. Six independent")
w("  techniques were applied, layered cheapest to most expensive: schema")
w("  drift, row count, key-set diff, row-level hash, aggregate")
w("  reconciliation, and a datacompy field-level comparison.")
w()
crit = sev_counts.get("CRITICAL", 0)
high = sev_counts.get("HIGH", 0)
w(f"  The migration is NOT clean. {n_defects} defects were logged, including")
w(f"  {crit} critical and {high} high severity. The reconciliation caught")
w(f"  {detection_rate} of the deliberately injected defect classes,")
w("  confirming the framework detects loss, duplication, corruption,")
w("  numeric drift, taxonomy renaming, and schema drift.")
w()

w("2. SCOPE OF WORK")
w("-" * 70)
w("  Validated the end-to-end data flow:")
w("    Source table  ->  Migration  ->  Target table  ->  Reconciliation")
w("  Techniques applied:")
w("    - Row count reconciliation (totals, delta, % variance)")
w("    - Schema drift detection (columns added / dropped / retyped)")
w("    - Key-set reconciliation (lost / injected / duplicated keys)")
w("    - Row-level hash reconciliation (MD5 over shared columns)")
w("    - Aggregate reconciliation (sum/min/max/distinct — silent drift)")
w("    - datacompy field-level diff (column-by-column on a sample)")
w()

w("3. KEY FINDINGS")
w("-" * 70)
w("  3.1 Compensating errors in the net row count")
w("      Rows were both lost AND duplicated; the net row delta understates")
w("      the true scale. A single check (row count alone) would have")
w("      misrepresented the migration as nearly complete. Reconciliation")
w("      must be layered: counts -> key-sets -> hashes -> field-level.")
w()
w("  3.2 Value-level corruption invisible to row counts")
w("      Aggregate and hash reconciliation surfaced changed values in rows")
w("      present in both source and target. Row counts and key-sets cannot")
w("      see this class of defect — only value-level checks can.")
w()
w("  3.3 Schema drift")
w("      The target schema differs from source. Added migration-metadata")
w("      columns are expected and logged LOW; a dropped business column is")
w("      a CRITICAL defect and logged as such.")
w()
w("  3.4 Taxonomy / categorical drift")
w("      Categorical values were renamed between source and target. This")
w("      silently breaks downstream grouping/joins on those values and is")
w("      only caught by aggregate/distinct checks.")
w()

w("4. RECONCILIATION SCORECARD")
w("-" * 70)
w("  Injected defect classes vs what QA detected (manifest = ground truth):")
w()
for cls, hit in scorecard:
    w(f"    [{'x' if hit else ' '}] {cls:<16} {'CAUGHT' if hit else 'MISSED'}")
w()
w(f"    Detection rate: {detection_rate}")
w()

w("5. DEFECT DETAIL (by severity)")
w("-" * 70)
w("  Severity rollup:")
for sev, n in sev_counts.items():
    w(f"    {sev:<9}: {n}")
w()
w("  Defect-type rollup:")
for row in type_rows:
    w(f"    {row['defect_type']:<28}: {row['n']}")
w()
w("  Individual defects:")
w()
for d in detail_rows:
    w(f"  [{d['severity']}] {d['defect_id']} — {d['defect_type']}")
    w(f"        column: {d['column_name']}")
    if d["source_value"] is not None:
        w(f"        source: {str(d['source_value'])[:64]}")
    if d["target_value"] is not None:
        w(f"        target: {str(d['target_value'])[:64]}")
    hyp = (d["root_cause_hypothesis"] or "").strip()
    if hyp:
        words, line = hyp.split(), ""
        for word in words:
            if len(line) + len(word) + 1 > 60:
                w(f"        {line}")
                line = word
            else:
                line = f"{line} {word}".strip()
        if line:
            w(f"        {line}")
    w()

w("6. RECOMMENDED REMEDIATION")
w("-" * 70)
w("  - Lost keys: investigate the migration filter that dropped rows;")
w("    re-migrate the missing key set and re-reconcile.")
w("  - Duplicated keys: the loader is not idempotent. Replace blind append")
w("    with a MERGE/upsert keyed on the business key; de-duplicate target.")
w("  - Value corruption: trace the transformation that mutated shared")
w("    columns; add a field-level check as a pre-promotion gate.")
w("  - Taxonomy drift: pin a categorical mapping table; validate distinct")
w("    values against it before promotion.")
w("  - Schema drift: confirm dropped columns are intentional; if not,")
w("    restore them. Gate promotion on a schema-parity check.")
w()
w("  QA owns detection, classification, and the verdict — not the repair.")
w("  Each defect is logged with a root-cause hypothesis for engineering to")
w("  action. Re-run this reconciliation after remediation to confirm.")
w()

w("7. CONCLUSION")
w("-" * 70)
w("  The migration requires remediation before promotion. The layered")
w("  reconciliation framework classified each defect by type and severity")
w("  rather than relying on a single signal. The net row count alone would")
w("  have been misleading; only the full layered approach surfaced the")
w("  true state of the migration.")
w()
w("=" * 70)
w(f"  END OF REPORT — {RUN_ID}")
w("=" * 70)

report_text = "\n".join(lines)
print(report_text)

  MIGRATION RECONCILIATION REPORT
  Run ID        : MIG-20260518-031539-70BA48
  Generated     : 2026-05-19 16:30:01 UTC
  Source        : workspace.fieldops_raw.cr_businesses_source
  Target        : workspace.fieldops_raw.cr_businesses_target
  Defects logged: 13
  Detection rate: 6/6 injected defect classes

1. EXECUTIVE SUMMARY
----------------------------------------------------------------------
  A source-to-target reconciliation validated the migration of the CR
  businesses registry into the Databricks lakehouse. Six independent
  techniques were applied, layered cheapest to most expensive: schema
  drift, row count, key-set diff, row-level hash, aggregate
  reconciliation, and a datacompy field-level comparison.

  The migration is NOT clean. 13 defects were logged, including
  2 critical and 5 high severity. The reconciliation caught
  6/6 of the deliberately injected defect classes,
  confirming the framework detects loss, duplication, corruption,
  numeric drift, taxonomy 

## Persist the report
Appends the report as one row to `dq.reconciliation_reports`, keyed by run_id and
timestamped — the same audit-trail principle used in regulated (PCI/SOX)
environments. Append-only so every run's report is retained and versioned.

In [0]:
report_row = spark.createDataFrame(
    [(
        RUN_ID, now_utc, src_tbl, tgt_tbl, n_defects,
        sev_counts.get("CRITICAL", 0), sev_counts.get("HIGH", 0),
        sev_counts.get("MEDIUM", 0), sev_counts.get("LOW", 0),
        detection_rate, report_text,
    )],
    schema=(
        "run_id string, generated_at string, source_table string, "
        "target_table string, total_defects int, critical int, high int, "
        "medium int, low int, detection_rate string, report_text string"
    ),
)

(
    report_row.write
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(REPORT_TBL)
)

print(f"Report persisted: {REPORT_TBL}")
print(f"  run_id         : {RUN_ID}")
print(f"  total defects  : {n_defects}")
print(f"  detection rate : {detection_rate}")

Report persisted: workspace.fieldops_dq.reconciliation_reports
  run_id         : MIG-20260518-031539-70BA48
  total defects  : 13
  detection rate : 6/6


## Verify
Reads the report back to confirm it persisted and is queryable.

In [0]:
(
    spark.table(REPORT_TBL)
    .filter(F.col("run_id") == RUN_ID)
    .select("run_id", "generated_at", "total_defects",
            "critical", "high", "medium", "low", "detection_rate")
    .show(truncate=False)
)

print("07 complete — reconciliation report generated, printed, and persisted.")

+--------------------------+-----------------------+-------------+--------+----+------+---+--------------+
|run_id                    |generated_at           |total_defects|critical|high|medium|low|detection_rate|
+--------------------------+-----------------------+-------------+--------+----+------+---+--------------+
|MIG-20260518-031539-70BA48|2026-05-19 16:30:01 UTC|13           |2       |5   |4     |2  |6/6           |
+--------------------------+-----------------------+-------------+--------+----+------+---+--------------+

07 complete — reconciliation report generated, printed, and persisted.
